### Set up environment

In [1]:
%pip install google-cloud-aiplatform python-dotenv fastapi uvicorn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### author

In [ ]:
!gcloud auth login

!gcloud auth application-default login

In [2]:
import os
from dotenv import load_dotenv
from google.cloud import aiplatform

# Load các cấu hình từ file .env bạn vừa sửa
load_dotenv(dotenv_path="../.env")

PROJECT_ID = os.environ.get("GCP_PROJECT_ID")
REGION = os.environ.get("GCP_REGION")
BUCKET_URI = os.environ.get("GCP_BUCKET_URI")

print(f"Project ID: {PROJECT_ID}")
print(f"Region: {REGION}")
print(f"Bucket: {BUCKET_URI}")

# Khởi tạo SDK
aiplatform.init(project=PROJECT_ID, location=REGION)


Project ID: graphic-theory-497611-t7
Region: us-central1
Bucket: gs://graphic-theory-497611-t7-bucket


### Deep learning container

In [3]:
model_garden  = aiplatform.Model.upload(
    display_name="hf-ticmastery",
    serving_container_image_uri="us-docker.pkg.dev/deeplearning-platform-release/gcs-fuse/huggingface-text-generation-inference.1-3.py310",
    serving_container_environment_variables={
        "HF_MODEL_ID": "Qwen/Qwen2.5-0.5B-Instruct",
        "HF_TASK": "text-generation",
        "MAX_INPUT_LENGTH": "1024",
        "MAX_TOTAL_TOKENS": "2048"
    }
)

endpoint_garden  = aiplatform.Endpoint.create(display_name="hf-ticmastery-endpoint")

model_garden .deploy(
    endpoint=endpoint_garden ,
    machine_type="n1-standard-2",
    min_replica_count=1,
    max_replica_count=1
)

DefaultCredentialsError: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information.

### Custom Serving Container

In [ ]:
# download weight to GCS
!gcloud storage cp -r ../notebook/qwen-2.5b-finetuned-qlora {BUCKET_URI}/qwen-2.5b-finetuned-qlora

# Create Artifact Registry Repository for Custom Model
!gcloud artifacts repositories create custom-model-qwen \
    --repository-format=docker \
    --location={REGION} \
    --description="Custom Model Repository for Qwen"

# Configure Docker to use Artifact Registry
!gcloud auth configure-docker {REGION}-docker.pkg.dev --quiet

# Build Docker image
!docker build -t custom-model-qwen:latest -f ../Dockerfile ..

# Tag Docker image
!docker tag custom-model-qwen:latest {REGION}-docker.pkg.dev/{PROJECT_ID}/custom-model-qwen:v1
# Push Docker image to Artifact Registry
!docker push {REGION}-docker.pkg.dev/{PROJECT_ID}/custom-model-qwen:v1

In [ ]:
model_custom = aiplatform.Model.upload(
    display_name="hf-ticmastery-custom",
    artifact_uri=f"{BUCKET_URI}/qwen-2.5b-finetuned-qlora",
    serving_container_image_uri=f"{REGION}-docker.pkg.dev/{PROJECT_ID}/custom-model-qwen:v1",
    serving_container_ports=[8080],
    serving_container_environment_variables={
        "GCS_URI": f"{BUCKET_URI}/qwen-2.5b-finetuned-qlora"
    }
)

endpoint_custom  = aiplatform.Endpoint.create(display_name="hf-ticmastery-custom-endpoint")

model_custom.deploy(
    endpoint=endpoint_custom ,
    machine_type="n1-standard-2",
    min_replica_count=1,
    max_replica_count=1
)

### Test

In [ ]:
response = endpoint_custom.predict(
    instances=["What is the capital of France?"]
)
print(response.predictions)

In [ ]:
endpoint_custom.undeploy_all()
endpoint_custom.delete()
endpoint_garden.undeploy_all()
endpoint_garden.delete()
print("Successfully undeployed and deleted the active endpoint.")

# ENDPOINT_ID = "projects/ticmastery/locations/us-central1/endpoints/[YOUR_ENDPOINT_NUMERIC_ID]"
# print(f"Deleting manual Endpoint ID: {ENDPOINT_ID}")
# endpoint_to_delete = aiplatform.Endpoint(endpoint_name=ENDPOINT_ID)
# endpoint_to_delete.undeploy_all()
# endpoint_to_delete.delete()